In [ ]:
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALSModel
from pyspark.sql.functions import explode, col

os.makedirs("/workspace/outputs/recommendations", exist_ok=True)

spark = (
    SparkSession.builder
    .appName("08_Recommendation_Result_Demo")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
    .config("spark.executor.memory", "3g")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "64")
    .getOrCreate()
)

HDFS_RAW = "hdfs://namenode:8020/netflix-recsys/raw/ml-25m"
HDFS_MODELS = "hdfs://namenode:8020/netflix-recsys/models"
HDFS_OUTPUTS = "hdfs://namenode:8020/netflix-recsys/outputs"

movies = spark.read.parquet(f"{HDFS_RAW}/movies")
als_model = ALSModel.load(f"{HDFS_MODELS}/als_model")

In [ ]:
def recommend_for_user(user_id: int, top_n: int = 10):
    user_df = spark.createDataFrame([(user_id,)], ["userId"])
    recs = als_model.recommendForUserSubset(user_df, top_n)

    result = (
        recs
        .select("userId", explode("recommendations").alias("rec"))
        .select(
            "userId",
            col("rec.movieId").alias("movieId"),
            col("rec.rating").alias("predicted_rating")
        )
        .join(movies, "movieId", "left")
        .orderBy(col("predicted_rating").desc())
    )

    pdf = result.toPandas()
    out_path = f"/workspace/outputs/recommendations/user_{user_id}_top_{top_n}.csv"
    pdf.to_csv(out_path, index=False)
    return pdf

recommend_for_user(1, 10)

In [ ]:
ratings = spark.read.parquet(f"{HDFS_RAW}/ratings")

def show_user_liked_movies(user_id: int, limit: int = 10):
    liked = (
        ratings
        .filter((col("userId") == user_id) & (col("rating") >= 4.0))
        .join(movies, "movieId", "left")
        .orderBy(col("rating").desc())
        .limit(limit)
    )
    return liked.toPandas()

show_user_liked_movies(1, 10)